# Pretrained 80/20 Model Trials and Figures
Loads `../output/models/hypertuning/*_8020_best_model.pkl`, runs three stratified bootstrap trials on the official 20% test set, and creates publication-ready combined and per-model figures. No model training is performed.

In [1]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import json
import numpy as np
import pandas as pd
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, matthews_corrcoef, confusion_matrix,
    roc_curve, auc, ConfusionMatrixDisplay
)
from sklearn.inspection import permutation_importance

try:
    import shap
    SHAP_AVAILABLE = True
except Exception as exc:
    SHAP_AVAILABLE = False
    print(f'SHAP unavailable; skipping SHAP plots: {exc}')

RANDOM_SEEDS = [42, 101, 202]
MODEL_DIR = Path('../output/models/hypertuning')
FIGURE_DIR = Path('../figures/pretrained_8020_publication')
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(context='paper', style='whitegrid', font_scale=1.35)
plt.rcParams.update({'savefig.dpi': 600, 'pdf.fonttype': 42, 'ps.fonttype': 42})


In [2]:
def display(obj=None, *args, **kwargs):
    try:
        print(obj.to_string() if hasattr(obj, 'to_string') else obj)
    except Exception:
        print(repr(obj))


def savefig(fig, name):
    fig.savefig(FIGURE_DIR / f'{name}.png', bbox_inches='tight')
    fig.savefig(FIGURE_DIR / f'{name}.pdf', bbox_inches='tight')
    plt.close(fig)


def load_split(split_ratio='8020', base_dir='../data/split'):
    base = Path(base_dir) / split_ratio
    X_train = pd.read_csv(base / 'x_train.csv', index_col=0).select_dtypes(include=[np.number])
    y_train = pd.read_csv(base / 'y_train.csv', index_col=0).iloc[:, 0].astype(int)
    X_test = pd.read_csv(base / 'x_test.csv', index_col=0)[X_train.columns].select_dtypes(include=[np.number])
    y_test = pd.read_csv(base / 'y_test.csv', index_col=0).iloc[:, 0].astype(int)
    return X_train.replace([np.inf, -np.inf], np.nan), y_train, X_test.replace([np.inf, -np.inf], np.nan), y_test


def discover_models(model_dir=MODEL_DIR):
    paths = sorted(model_dir.glob('*_8020_best_model.pkl'))
    return {path.name.replace('_8020_best_model.pkl', ''): joblib.load(path) for path in paths}


def predict_scores(model, X):
    y_pred = model.predict(X)
    if hasattr(model, 'predict_proba'):
        y_score = model.predict_proba(X)[:, 1]
    elif hasattr(model, 'decision_function'):
        raw = model.decision_function(X)
        y_score = (raw - raw.min()) / (raw.max() - raw.min() + 1e-12)
    else:
        y_score = y_pred.astype(float)
    return y_pred, y_score


def compute_metrics(y_true, y_pred, y_score):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'specificity': tn / (tn + fp) if (tn + fp) else np.nan,
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_true, y_score),
        'average_precision': average_precision_score(y_true, y_score),
        'mcc': matthews_corrcoef(y_true, y_pred),
    }


def stratified_bootstrap_indices(y, seed):
    rng = np.random.default_rng(seed)
    y_array = np.asarray(y)
    indices = []
    for cls in np.unique(y_array):
        cls_idx = np.where(y_array == cls)[0]
        indices.append(rng.choice(cls_idx, size=len(cls_idx), replace=True))
    out = np.concatenate(indices)
    rng.shuffle(out)
    return out


def transform_for_final_estimator(model, X):
    transformed = X.copy()
    feature_names = np.array(X.columns, dtype=object)
    if not hasattr(model, 'steps'):
        return transformed, feature_names, model
    for step_name, step in model.steps[:-1]:
        if step == 'passthrough' or 'sampler' in step_name.lower() or 'under' in step_name.lower():
            continue
        if hasattr(step, 'transform'):
            transformed = step.transform(transformed)
        if hasattr(step, 'get_feature_names_out'):
            try:
                feature_names = step.get_feature_names_out(feature_names)
            except Exception:
                pass
    return transformed, feature_names, model.steps[-1][1]


## Load Models and Create Trial Tables

In [4]:
X_train, y_train, X_test, y_test = load_split('8020')
models = discover_models(MODEL_DIR)
if not models:
    raise FileNotFoundError(f'No *_8020_best_model.pkl files found in {MODEL_DIR}')
print(f'Loaded pretrained models: {list(models.keys())}')

feature_manifest = {'features': X_train.columns.tolist()}
(MODEL_DIR / 'feature_names_existing_8020.json').write_text(json.dumps(feature_manifest, indent=2))
joblib.dump(X_train.columns.tolist(), MODEL_DIR / 'feature_names_existing_8020.pkl')

official_rows = []
prediction_frames = []
for model_name, model in models.items():
    y_pred, y_score = predict_scores(model, X_test)
    metrics = compute_metrics(y_test, y_pred, y_score)
    official_rows.append({'model': model_name, 'split': '8020_official_pretrained', **metrics})
    prediction_frames.append(pd.DataFrame({'model': model_name, 'index': X_test.index, 'y_true': y_test.values, 'y_pred': y_pred, 'y_score': y_score}))

official = pd.DataFrame(official_rows).sort_values('roc_auc', ascending=False)
predictions = pd.concat(prediction_frames, ignore_index=True)
official.to_csv(MODEL_DIR / 'results_existing_8020_official.csv', index=False)
predictions.to_csv(MODEL_DIR / 'predictions_all_models_existing_8020.csv', index=False)

trial_rows = []
trial_prediction_frames = []
for trial_id, seed in enumerate(RANDOM_SEEDS, start=1):
    boot_idx = stratified_bootstrap_indices(y_test.values, seed)
    X_trial = X_test.iloc[boot_idx]
    y_trial = y_test.iloc[boot_idx]
    for model_name, model in models.items():
        y_pred, y_score = predict_scores(model, X_trial)
        metrics = compute_metrics(y_trial, y_pred, y_score)
        trial_rows.append({'trial': trial_id, 'seed': seed, 'model': model_name, **metrics})
        trial_prediction_frames.append(pd.DataFrame({'trial': trial_id, 'seed': seed, 'model': model_name, 'index': X_trial.index, 'y_true': y_trial.values, 'y_pred': y_pred, 'y_score': y_score}))

trials = pd.DataFrame(trial_rows)
trial_predictions = pd.concat(trial_prediction_frames, ignore_index=True)
summary = trials.groupby('model').agg(['mean', 'std'])
summary.columns = [f'{metric}_{stat}' for metric, stat in summary.columns]
summary = summary.reset_index().sort_values('roc_auc_mean', ascending=False)

trials.to_csv(MODEL_DIR / 'trials_results_3runs_existing_8020.csv', index=False)
trial_predictions.to_csv(MODEL_DIR / 'trial_predictions_3runs_existing_8020.csv', index=False)
summary.to_csv(MODEL_DIR / 'trials_summary_mean_std_existing_8020.csv', index=False)

display(official)
display(summary)


Loaded pretrained models: ['ET', 'KNN', 'LGBM', 'LogReg', 'RF', 'XGB']
    model                     split  accuracy  balanced_accuracy  precision    recall  specificity        f1   roc_auc  average_precision       mcc
0      ET  8020_official_pretrained  0.856841           0.860994   0.941516  0.851080     0.870909  0.894016  0.907543           0.963041  0.683448
4      RF  8020_official_pretrained  0.851030           0.855825   0.939519  0.844378     0.867273  0.889412  0.906865           0.963744  0.672157
5     XGB  8020_official_pretrained  0.854200           0.859133   0.941274  0.847357     0.870909  0.891850  0.906510           0.962133  0.678834
2    LGBM  8020_official_pretrained  0.849445           0.852025   0.935750  0.845867     0.858182  0.888541  0.906296           0.961508  0.666466
1     KNN  8020_official_pretrained  0.841521           0.840535   0.927109  0.842889     0.838182  0.882995  0.884807           0.939685  0.646203
3  LogReg  8020_official_pretrained  0.80

## Combined Figures

In [4]:
order = summary.sort_values('roc_auc_mean', ascending=False)['model'].tolist()
metric_panels = ['accuracy', 'balanced_accuracy', 'precision', 'recall', 'specificity', 'f1', 'roc_auc', 'average_precision', 'mcc']
plot_df = summary.set_index('model').loc[order]

fig, axes = plt.subplots(3, 3, figsize=(18, 14), constrained_layout=True)
for ax, metric in zip(axes.ravel(), metric_panels):
    means = plot_df[f'{metric}_mean']
    stds = plot_df[f'{metric}_std'].fillna(0)
    ax.bar(order, means, yerr=stds, capsize=5, color='#4C78A8', edgecolor='black', linewidth=0.7)
    ax.set_title(metric.replace('_', ' ').title(), fontweight='bold')
    ax.set_ylabel('Mean ± SD')
    ax.tick_params(axis='x', rotation=35)
    if metric != 'mcc':
        ax.set_ylim(0, 1.02)
fig.suptitle('Pretrained 80/20 BBB Models: Bootstrap Trial Mean ± SD', fontsize=18, fontweight='bold')
savefig(fig, 'combined_metrics_mean_std_panel')

radar_metrics = ['accuracy', 'balanced_accuracy', 'precision', 'recall', 'specificity', 'f1', 'roc_auc']
angles = np.linspace(0, 2 * np.pi, len(radar_metrics), endpoint=False).tolist()
angles += angles[:1]
fig = plt.figure(figsize=(10, 10))
ax = plt.subplot(111, polar=True)
for model in order:
    row = summary[summary['model'] == model].iloc[0]
    values = [row[f'{m}_mean'] for m in radar_metrics] + [row[f'{radar_metrics[0]}_mean']]
    ax.plot(angles, values, linewidth=2, label=model)
    ax.fill(angles, values, alpha=0.08)
ax.set_xticks(angles[:-1])
ax.set_xticklabels([m.replace('_', ' ').title() for m in radar_metrics], fontsize=12)
ax.set_ylim(0, 1)
ax.set_title('Pretrained 80/20 BBB Models\nSpider Chart of Mean Metrics', fontsize=17, fontweight='bold', pad=28)
ax.legend(loc='upper right', bbox_to_anchor=(1.28, 1.12), frameon=True)
savefig(fig, 'combined_spider_radar_metrics')

fig, ax = plt.subplots(figsize=(8, 7))
for model in order:
    g = predictions[predictions['model'] == model]
    fpr, tpr, _ = roc_curve(g['y_true'], g['y_score'])
    ax.plot(fpr, tpr, linewidth=2, label=f'{model} (AUC={auc(fpr, tpr):.3f})')
ax.plot([0, 1], [0, 1], color='gray', linestyle='--', linewidth=1)
ax.set_title('Pretrained 80/20 BBB Models: ROC Curves', fontweight='bold')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend(loc='lower right', frameon=True)
savefig(fig, 'combined_roc_curves')


## Individual Per-Model Figures

In [5]:
individual_metrics = ['accuracy', 'balanced_accuracy', 'precision', 'recall', 'specificity', 'f1', 'roc_auc', 'average_precision']
for model in order:
    row = summary[summary['model'] == model].iloc[0]
    labels = [m.replace('_', ' ').title() for m in individual_metrics]
    means = [row[f'{m}_mean'] for m in individual_metrics]
    stds = [row[f'{m}_std'] for m in individual_metrics]

    fig, ax = plt.subplots(figsize=(10, 5.8))
    ax.bar(labels, means, yerr=stds, capsize=5, color='#4C78A8', edgecolor='black', linewidth=0.7)
    ax.set_title(f'{model}: Bootstrap Trial Mean Metrics ± SD', fontweight='bold')
    ax.set_ylabel('Mean ± SD')
    ax.set_ylim(0, 1.05)
    ax.tick_params(axis='x', rotation=35)
    savefig(fig, f'{model}_individual_metric_profile')

    g = predictions[predictions['model'] == model]
    fig, ax = plt.subplots(figsize=(7, 6))
    fpr, tpr, _ = roc_curve(g['y_true'], g['y_score'])
    ax.plot(fpr, tpr, linewidth=2.5, color='#E45756', label=f'AUC={auc(fpr, tpr):.3f}')
    ax.plot([0, 1], [0, 1], color='gray', linestyle='--', linewidth=1)
    ax.set_title(f'{model}: ROC Curve', fontweight='bold')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.legend(loc='lower right', frameon=True)
    savefig(fig, f'{model}_individual_roc_curve')

    fig, ax = plt.subplots(figsize=(5.7, 5.2))
    cm = confusion_matrix(g['y_true'], g['y_pred'], labels=[0, 1])
    ConfusionMatrixDisplay(cm, display_labels=['BBB-', 'BBB+']).plot(ax=ax, cmap='Blues', colorbar=False, values_format='d')
    ax.set_title(f'{model}: Confusion Matrix', fontweight='bold')
    savefig(fig, f'{model}_individual_confusion_matrix')


## Feature Importance and SHAP

In [6]:
sample_X = X_test.sample(min(200, len(X_test)), random_state=42)
sample_y = y_test.loc[sample_X.index]
order = summary.sort_values('roc_auc_mean', ascending=False)['model'].tolist()
for model in order:
    estimator = models[model]
    
    # Try to get built-in feature importance first (fast for tree-based models)
    if hasattr(estimator, 'feature_importances_'):
        imp_values = estimator.feature_importances_
        imp = pd.DataFrame({'feature': sample_X.columns, 'importance': imp_values})
    elif hasattr(estimator, 'coef_'):
        # For linear models, use absolute coefficient values
        imp_values = np.abs(estimator.coef_).ravel()[:len(sample_X.columns)]
        imp = pd.DataFrame({'feature': sample_X.columns, 'importance': imp_values})
    else:
        # Skip if model doesn't have simple feature importance
        print(f'Skipping feature importance for {model} (no built-in importance)')
        continue
    
    # Normalize importance to 0-1 range for comparison
    imp['importance'] = imp['importance'] / imp['importance'].max()
    imp = imp.sort_values('importance', ascending=False).head(15).sort_values('importance')
    
    # Create simple bar plot
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.barh(imp['feature'], imp['importance'], color='#4C78A8', edgecolor='black', linewidth=0.5)
    ax.set_title(f'{model}: Top Feature Importance', fontweight='bold', fontsize=12)
    ax.set_xlabel('Normalized Importance')
    savefig(fig, f'{model}_feature_importance')

summary.to_csv(FIGURE_DIR / 'pretrained_8020_metrics_mean_std_table.csv', index=False)
official.to_csv(FIGURE_DIR / 'pretrained_8020_official_metrics_table.csv', index=False)
print(f'Feature importance figures and tables saved to: {FIGURE_DIR}')

Skipping feature importance for RF (no built-in importance)
Skipping feature importance for ET (no built-in importance)
Skipping feature importance for LGBM (no built-in importance)
Skipping feature importance for XGB (no built-in importance)
Skipping feature importance for KNN (no built-in importance)
Skipping feature importance for LogReg (no built-in importance)
Feature importance figures and tables saved to: ../figures/pretrained_8020_publication


In [7]:
print("test")

test


In [8]:
# Highly correlated descriptor pairs for ET model
from scipy.stats import pearsonr

et_model = models['ET']

# Transform features through the ET pipeline to get selected descriptors
transformed_X = X_test.copy()
feature_names = np.array(X_test.columns, dtype=object)

if hasattr(et_model, 'steps'):
    for step_name, step in et_model.steps[:-1]:
        if step == 'passthrough' or 'sampler' in step_name.lower():
            continue
        if hasattr(step, 'transform'):
            transformed_X = step.transform(transformed_X)
        if hasattr(step, 'get_feature_names_out'):
            try:
                feature_names = step.get_feature_names_out(feature_names)
            except Exception:
                pass

# Create dataframe with selected features
et_features_df = pd.DataFrame(transformed_X, columns=feature_names)

# Calculate correlation matrix
corr_matrix = et_features_df.corr(method='pearson')

# Extract highly correlated pairs (excluding diagonal and duplicates)
corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        feat1 = corr_matrix.columns[i]
        feat2 = corr_matrix.columns[j]
        corr_val = corr_matrix.iloc[i, j]
        abs_corr = abs(corr_val)
        if abs_corr >= 0.7:  # High correlation threshold
            corr_pairs.append({
                'Descriptor_1': feat1,
                'Descriptor_2': feat2,
                'Pearson_Correlation': corr_val,
                'Abs_Correlation': abs_corr
            })

# Create and sort table
corr_pairs_table = pd.DataFrame(corr_pairs).sort_values('Abs_Correlation', ascending=False)

# Display and save
display(corr_pairs_table)
corr_pairs_table.to_csv(FIGURE_DIR / 'ET_highly_correlated_descriptors.csv', index=False)
print(f'\n✓ Found {len(corr_pairs_table)} descriptor pairs with correlation >= 0.7')
print(f'✓ Saved to: {FIGURE_DIR / "ET_highly_correlated_descriptors.csv"}')

           Descriptor_1       Descriptor_2  Pearson_Correlation  Abs_Correlation
2334          ETA_Psi_1         ETA_dPsi_A            -0.999026         0.999026
2541         topoRadius            SpMAD_D             0.994640         0.994640
41                 apol              ATS0p             0.986342         0.986342
2550            SpMax_D              WPATH             0.977468         0.977468
1245            VR2_Dzv            VR2_Dzp             0.977144         0.977144
2220             nddssS           minddssS            -0.977144         0.977144
1188            VR1_DzZ            VR1_Dzv             0.975352         0.975352
1469         SpMin5_Bhm         SpMin5_Bhv             0.970101         0.970101
1541         SpMin8_Bhm         SpMin8_Bhv             0.969978         0.969978
1519         SpMin7_Bhm         SpMin7_Bhv             0.969779         0.969779
1248            SM1_Dze               nHBa             0.969624         0.969624
613              AATS0s     

In [11]:
# Feature correlation heatmap for top 15 important ET descriptors
et_model = models['ET']

# Extract final estimator and transformed features from the pipeline
et_transformed, et_feature_names, et_final_estimator = transform_for_final_estimator(et_model, X_test)
et_features_full_df = pd.DataFrame(et_transformed, columns=et_feature_names)

# Get feature importances from the final estimator
if hasattr(et_final_estimator, 'feature_importances_'):
    importances = et_final_estimator.feature_importances_
    feat_imp = pd.DataFrame({
        'feature': et_feature_names,
        'importance': importances
    }).sort_values('importance', ascending=False)
    
    # Get top 15 features
    top_15_features = feat_imp.head(15)['feature'].tolist()
    top_15_data = et_features_full_df[top_15_features]
    
    # Calculate correlation matrix
    corr_matrix_top15 = top_15_data.corr(method='pearson')
    
    # Create heatmap
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(corr_matrix_top15, annot=True, fmt='.2f', cmap='coolwarm', center=0,
                square=True, linewidths=0.5, cbar_kws={'label': 'Pearson Correlation'},
                vmin=-1, vmax=1, ax=ax)
    ax.set_title('ET Model: Top 15 Features by Importance - Correlation Heatmap', 
                 fontweight='bold', fontsize=14, pad=20)
    plt.tight_layout()
    savefig(fig, 'ET_top15_features_correlation_heatmap')
    
    # Also save feature importance ranking
    feat_imp.head(15).to_csv(FIGURE_DIR / 'ET_top15_features_importance_ranking.csv', index=False)
    
    print(f'\n✓ Top 15 features by importance for ET model:')
    display(feat_imp.head(15))
    print(f'✓ Heatmap saved to: {FIGURE_DIR / "ET_top15_features_correlation_heatmap.png"}')
else:
    print('ET final estimator does not have feature_importances_ attribute')


✓ Top 15 features by importance for ET model:
        feature  importance
696   maxHBint5    0.010806
702     maxHsOH    0.010197
699   maxHBint8    0.009678
611     minHsOH    0.009275
928     TopoPSA    0.009247
693   maxHBint2    0.008406
700   maxHBint9    0.006706
701  maxHBint10    0.006383
19        ATS0s    0.006135
697   maxHBint6    0.005852
536       SHsOH    0.005656
644      minsOH    0.005464
426        nHBd    0.005008
648       minsF    0.005005
698   maxHBint7    0.004888
✓ Heatmap saved to: ../figures/pretrained_8020_publication/ET_top15_features_correlation_heatmap.png


In [12]:
# Table of descriptor pairs and correlations for top 15 ET features
if hasattr(et_final_estimator, 'feature_importances_'):
    # Extract descriptor pairs and their correlations
    descriptor_corr_pairs = []
    
    for i in range(len(corr_matrix_top15.columns)):
        for j in range(i+1, len(corr_matrix_top15.columns)):
            desc1 = corr_matrix_top15.columns[i]
            desc2 = corr_matrix_top15.columns[j]
            corr_val = corr_matrix_top15.iloc[i, j]
            
            descriptor_corr_pairs.append({
                'Descriptor_1': desc1,
                'Descriptor_2': desc2,
                'Pearson_Correlation': corr_val,
                'Abs_Correlation': abs(corr_val)
            })
    
    # Create DataFrame and sort by absolute correlation (strongest first)
    desc_corr_df = pd.DataFrame(descriptor_corr_pairs).sort_values('Abs_Correlation', ascending=False)
    
    # Save to CSV
    desc_corr_df.to_csv(FIGURE_DIR / 'ET_top15_descriptor_correlations.csv', index=False)
    
    print(f'\n✓ Descriptor correlations for top 15 ET features:')
    display(desc_corr_df)
    print(f'✓ Table saved to: {FIGURE_DIR / "ET_top15_descriptor_correlations.csv"}')
else:
    print('Could not generate descriptor correlation table')


✓ Descriptor correlations for top 15 ET features:
    Descriptor_1 Descriptor_2  Pearson_Correlation  Abs_Correlation
15       maxHsOH      minHsOH             0.876391         0.876391
57       TopoPSA         nHBd             0.808633         0.808633
53       TopoPSA        ATS0s             0.806236         0.806236
22       maxHsOH        SHsOH             0.796594         0.796594
23       maxHsOH       minsOH             0.740702         0.740702
96         SHsOH         nHBd             0.737656         0.737656
52       TopoPSA   maxHBint10             0.658437         0.658437
28     maxHBint8      TopoPSA             0.654321         0.654321
51       TopoPSA    maxHBint9             0.638045         0.638045
87         ATS0s         nHBd             0.631320         0.631320
81    maxHBint10         nHBd             0.617995         0.617995
54       TopoPSA    maxHBint6             0.612791         0.612791
45       minHsOH        SHsOH             0.602413         0.6024